# 第6章 高级数据操作

## 📌 学习目标

- INSERT ... ON DUPLICATE KEY UPDATE（Upsert）
- REPLACE INTO
- INSERT ... SELECT
- 批量操作优化
- LOAD DATA INFILE

---

In [ ]:
import os
import pymysql
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": "shop_db",
    "charset": "utf8mb4",
}

def run_query(sql, params=None):
    conn = pymysql.connect(**DB_CONFIG)
    try:
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

## 1. INSERT ... ON DUPLICATE KEY UPDATE

如果插入时主键或唯一键冲突，则执行更新操作（Upsert）。

```sql
INSERT INTO 表名 (列1, 列2)
VALUES (值1, 值2)
ON DUPLICATE KEY UPDATE
    列2 = VALUES(列2);
```

In [ ]:
# 创建演示表
execute_sql("DROP TABLE IF EXISTS inventory")
execute_sql("""
    CREATE TABLE inventory (
        product_id INT PRIMARY KEY,
        stock INT DEFAULT 0,
        last_updated DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
    )
""")

# 首次插入
execute_sql("INSERT INTO inventory (product_id, stock) VALUES (1, 100)")
print("首次插入:")
display(run_query("SELECT * FROM inventory"))

# 再次插入相同 product_id → 更新
execute_sql("""
    INSERT INTO inventory (product_id, stock) VALUES (1, 150)
    ON DUPLICATE KEY UPDATE stock = VALUES(stock)
""")
print("\nUpsert 后:")
display(run_query("SELECT * FROM inventory"))

In [ ]:
# 累加库存
execute_sql("""
    INSERT INTO inventory (product_id, stock) VALUES (1, 50)
    ON DUPLICATE KEY UPDATE stock = stock + VALUES(stock)
""")
print("累加库存后:")
run_query("SELECT * FROM inventory")

## 2. REPLACE INTO

如果主键或唯一键冲突，先删除旧行，再插入新行。

```sql
REPLACE INTO 表名 (列1, 列2) VALUES (值1, 值2);
```

In [ ]:
# REPLACE INTO
execute_sql("REPLACE INTO inventory (product_id, stock) VALUES (1, 200)")
print("REPLACE INTO 后:")
run_query("SELECT * FROM inventory")

## 3. INSERT ... SELECT

从查询结果插入数据。

In [ ]:
# 创建备份表
execute_sql("DROP TABLE IF EXISTS orders_backup")
execute_sql("CREATE TABLE orders_backup LIKE orders")

# 从查询结果插入
execute_sql("""
    INSERT INTO orders_backup
    SELECT * FROM orders WHERE status = '已完成'
""")

run_query("SELECT COUNT(*) AS 备份订单数 FROM orders_backup")

## 4. 批量操作优化

### 4.1 多值 INSERT vs 逐条 INSERT

In [ ]:
# 多值 INSERT（推荐）
execute_sql("DROP TABLE IF EXISTS batch_test")
execute_sql("CREATE TABLE batch_test (id INT PRIMARY KEY, name VARCHAR(50))")

execute_sql("""
    INSERT INTO batch_test (id, name) VALUES
    (1, 'A'), (2, 'B'), (3, 'C'), (4, 'D'), (5, 'E')
""")

run_query("SELECT * FROM batch_test")

### 4.2 批量更新

使用 CASE WHEN 实现批量条件更新。

In [ ]:
# 批量更新：根据订单状态设置不同的折扣
execute_sql("""
    UPDATE orders
    SET total_amount = total_amount * CASE status
        WHEN '已完成' THEN 1.0
        WHEN '已发货' THEN 0.95
        WHEN '待处理' THEN 0.90
        ELSE 1.0
    END
    WHERE order_id <= 5
""")

run_query("SELECT order_id, status, total_amount FROM orders WHERE order_id <= 5")

In [ ]:
# 还原数据
execute_sql("""
    UPDATE orders
    SET total_amount = total_amount / CASE status
        WHEN '已完成' THEN 1.0
        WHEN '已发货' THEN 0.95
        WHEN '待处理' THEN 0.90
        ELSE 1.0
    END
    WHERE order_id <= 5
""")

## 5. 清理

In [ ]:
execute_sql("DROP TABLE IF EXISTS inventory")
execute_sql("DROP TABLE IF EXISTS orders_backup")
execute_sql("DROP TABLE IF EXISTS batch_test")
print("清理完成")

## 📝 练习题

### 题目1（简单）
创建一个 `user_scores` 表（user_id PRIMARY KEY, score INT），插入一条数据，然后使用 ON DUPLICATE KEY UPDATE 更新分数。

### 题目2（中等）
使用 INSERT ... SELECT 将所有北京客户的订单复制到一个新表 `beijing_orders` 中。

### 题目3（中等）
使用 CASE WHEN 批量更新：将所有商品的价格按分类调整（电子产品 +10%，服装 -5%，其他不变）。

---

## ✅ 参考答案

In [ ]:
# 题目1：Upsert
execute_sql("DROP TABLE IF EXISTS user_scores")
execute_sql("CREATE TABLE user_scores (user_id INT PRIMARY KEY, score INT)")

# 首次插入
execute_sql("INSERT INTO user_scores (user_id, score) VALUES (1, 80)")
print("首次插入:")
display(run_query("SELECT * FROM user_scores"))

# Upsert 更新
execute_sql("""
    INSERT INTO user_scores (user_id, score) VALUES (1, 95)
    ON DUPLICATE KEY UPDATE score = VALUES(score)
""")
print("\n更新后:")
display(run_query("SELECT * FROM user_scores"))

execute_sql("DROP TABLE IF EXISTS user_scores")

In [ ]:
# 题目2：INSERT ... SELECT
execute_sql("DROP TABLE IF EXISTS beijing_orders")
execute_sql("CREATE TABLE beijing_orders LIKE orders")

execute_sql("""
    INSERT INTO beijing_orders
    SELECT o.*
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    WHERE c.city = '北京'
""")

run_query("SELECT * FROM beijing_orders")
execute_sql("DROP TABLE IF EXISTS beijing_orders")

In [ ]:
# 题目3：批量更新
execute_sql("""
    UPDATE products
    SET price = ROUND(price * CASE category_id
        WHEN 1 THEN 1.10  -- 电子产品 +10%
        WHEN 2 THEN 0.95  -- 服装 -5%
        ELSE 1.00
    END, 2)
""")

run_query("""
    SELECT c.category_name, p.product_name, p.price
    FROM products p
    INNER JOIN categories c ON p.category_id = c.category_id
    ORDER BY c.category_name, p.price DESC
""")